# Decisions

> ### Learning Objectives
>
> By the end of this chapter you should be able to work with:
>
> - How conditional statements alter the normal sequential flow of control
> - `if`, `if-else`, and `if-elif-else` statements and their indented code blocks
> - Identifying and correcting logical errors in conditional expressions
> - Nested conditional statements for modelling more complex decision logic
> - Why direct `==` comparison of floating-point values is unreliable
> - Safe float comparison using a threshold (with `abs`) and `math.isclose()`
> - String comparison by ordinal (ASCII/Unicode) value, and the `ord()` built-in
> - Translating real-world decision rules (bank overdraft, WNBA scoring) into readable conditional code

In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  SETUP — run this cell first.
#
#  It draws every figure used in this chapter and switches the notebook into
#  "show me every result" mode.  Everything it needs is right here: nothing to
#  install, nothing to download, no other files required.
#
#  (Curious what a figure is made of?  The drawing code is all below.)
# ══════════════════════════════════════════════════════════════════════
import io

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, Polygon, Circle
from matplotlib.lines import Line2D
from IPython.display import Image, display
from IPython.core.interactiveshell import InteractiveShell

# echo the value of *every* expression in a cell, the way the Python prompt
# does — many examples in this book show several results at once
InteractiveShell.ast_node_interactivity = "all"

# ------------------------------------------------------------- drawing ---

INK = "#1a1a1a"

MUTED = "#6b7280"

FILL = "#eef2f7"

ACCENT = "#2563eb"

WARM = "#b45309"

EDGE = "#334155"

def _frame(ax, xlim, ylim, title=None):
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.set_aspect("equal")
    ax.axis("off")
    if title:
        ax.set_title(title, fontsize=10, color=MUTED, pad=8)

def _box(ax, xy, text, w=2.6, h=0.9, fc=FILL, ec=EDGE, fs=9, bold=False):
    x, y = xy
    ax.add_patch(FancyBboxPatch(
        (x - w / 2, y - h / 2), w, h,
        boxstyle="round,pad=0.02,rounding_size=0.12",
        linewidth=1.3, facecolor=fc, edgecolor=ec, zorder=2))
    ax.text(x, y, text, ha="center", va="center", fontsize=fs, color=INK,
            zorder=3, fontweight="bold" if bold else "normal")
    return xy

def _diamond(ax, xy, text, w=3.0, h=1.5, fc="#fff7ed", ec=WARM, fs=9):
    x, y = xy
    ax.add_patch(Polygon(
        [(x, y + h / 2), (x + w / 2, y), (x, y - h / 2), (x - w / 2, y)],
        closed=True, linewidth=1.3, facecolor=fc, edgecolor=ec, zorder=2))
    ax.text(x, y, text, ha="center", va="center", fontsize=fs, color=INK, zorder=3)
    return xy

def _dot(ax, xy, r=0.09):
    ax.add_patch(Circle(xy, r, facecolor=EDGE, edgecolor=EDGE, zorder=4))
    return xy

def _arrow(ax, pts, label=None, label_at=0.5, label_off=(0.0, 0.18),
           color=EDGE, ha="center"):
    """Poly-line arrow through `pts` (elbow routing), head on the last segment."""
    xs = [p[0] for p in pts]
    ys = [p[1] for p in pts]
    ax.add_line(Line2D(xs[:-1] + [xs[-1]], ys[:-1] + [ys[-1]],
                       color=color, linewidth=1.3, zorder=1,
                       solid_capstyle="round"))
    ax.annotate("", xy=pts[-1], xytext=pts[-2],
                arrowprops=dict(arrowstyle="-|>", color=color, linewidth=1.3,
                                shrinkA=0, shrinkB=0), zorder=1)
    if label:
        i = max(0, min(len(pts) - 2, int(label_at * (len(pts) - 1))))
        mx = (pts[i][0] + pts[i + 1][0]) / 2 + label_off[0]
        my = (pts[i][1] + pts[i + 1][1]) / 2 + label_off[1]
        ax.text(mx, my, label, fontsize=8, color=MUTED, ha=ha, va="center")

def _line(ax, pts, color=EDGE):
    """Poly-line with no arrowhead — for merging branches into a shared rail."""
    ax.add_line(Line2D([p[0] for p in pts], [p[1] for p in pts], color=color,
                       linewidth=1.3, zorder=1, solid_capstyle="round"))

def _cellgrid(ax, values, origin=(0, 0), cw=1.0, ch=1.0, fs=13, fc="white"):
    """A row/table of boxed cells; `values` is a list of rows."""
    x0, y0 = origin
    for r, row in enumerate(values):
        for c, v in enumerate(row):
            x = x0 + c * cw
            y = y0 - r * ch
            ax.add_patch(plt.Rectangle((x, y - ch), cw, ch, facecolor=fc,
                                       edgecolor=EDGE, linewidth=1.2, zorder=2))
            ax.text(x + cw / 2, y - ch / 2, str(v), ha="center", va="center",
                    fontsize=fs, color=INK, family="monospace", zorder=3)

def _mockwindow(ax, w, h, title, body, titlebar="#d7dde5", face="#ffffff",
                fs=9, textcolor=INK):
    """A framed window with a title bar and monospaced body lines."""
    ax.add_patch(plt.Rectangle((0, 0), w, h, facecolor=face, edgecolor=EDGE,
                               linewidth=1.2, zorder=1))
    ax.add_patch(plt.Rectangle((0, h - 0.55), w, 0.55, facecolor=titlebar,
                               edgecolor=EDGE, linewidth=1.2, zorder=2))
    ax.text(0.2, h - 0.28, title, fontsize=9, va="center", color=INK, zorder=3)
    y = h - 1.05
    for line, colour in body:
        ax.text(0.25, y, line, fontsize=fs, va="center", family="monospace",
                color=colour or textcolor, zorder=3)
        y -= 0.5

def _index_grid(ax, items, top_label, side_label, fs=13):
    n = len(items)
    _cellgrid(ax, [items], origin=(0, 1), fs=fs)
    for i in range(n):
        ax.text(i + 0.5, 1.25, str(i), ha="center", va="bottom", fontsize=10,
                color=ACCENT)
    if top_label:
        ax.text(-0.25, 1.3, top_label, ha="right", va="bottom", fontsize=9,
                color=ACCENT)
    if side_label:
        ax.text(-0.25, 0.5, side_label, ha="right", va="center", fontsize=9,
                color=ACCENT)
    _frame(ax, (-6.4, n + 0.4), (-0.4, 2.0))

def _double_diamond(ax, stage=None):
    names = ["Understand", "Design", "Implement", "Evaluate"]
    # two diamonds: centres at x=2.6 and x=7.8, half-width 2.6, half-height 2.0
    for d, cx in enumerate((2.6, 7.8)):
        left, right, top, bot = cx - 2.6, cx + 2.6, 2.0, -2.0
        for half in (0, 1):
            name = names[2 * d + half]
            tri = ([(left, 0), (cx, top), (cx, bot)] if half == 0
                   else [(cx, top), (right, 0), (cx, bot)])
            on = (name == stage)
            ax.add_patch(Polygon(tri, closed=True, zorder=1,
                                 facecolor="#b9bfc7" if on else "#eceef1",
                                 edgecolor="none"))
            tx = cx - 1.3 if half == 0 else cx + 1.3
            ax.text(tx, 0, name, ha="center", va="center", fontsize=10,
                    color=INK if on else MUTED,
                    fontweight="bold" if on else "normal", zorder=3)
        ax.add_patch(Polygon([(left, 0), (cx, top), (right, 0), (cx, bot)],
                             closed=True, facecolor="none", edgecolor=INK,
                             linewidth=2.2, zorder=2))
        ax.plot([cx, cx], [top, bot], color=INK, linewidth=1.0, zorder=2)
    for x, y in ((0.0, 0.0), (5.2, 0.0), (10.4, 0.0)):
        ax.add_patch(Circle((x, y), 0.22, facecolor="#c9ced6", edgecolor=INK,
                            linewidth=1.2, zorder=4))
    ax.plot([-1.5, -0.22], [0, 0], color=INK, linewidth=1.6, zorder=2)
    ax.plot([10.62, 11.9], [0, 0], color=INK, linewidth=1.6, zorder=2)
    ax.plot([5.2, 5.2], [-0.22, -3.1], color=INK, linewidth=1.6, zorder=2)
    ax.text(-1.7, 0, "Problem", ha="right", va="center", fontsize=10)
    ax.text(12.1, 0, "Program", ha="left", va="center", fontsize=10)
    ax.text(5.2, -3.35, "Specification", ha="center", va="top", fontsize=10)
    _frame(ax, (-4.2, 14.2), (-4.2, 2.6))

def draw_flow_if_else(ax):
    """Replaces the TikZ flowchart in 05_Branching (if / else)."""
    ifc = _diamond(ax, (0, 3.2), "if condition:")
    tb = _box(ax, (-2.6, 1.4), "execute\n\"if\" block")
    fb = _box(ax, (2.6, 1.4), "execute\n\"else\" block")
    res = _box(ax, (0, -0.3), "code after the \"else\" block", w=4.6)
    _arrow(ax, [(-1.5, 3.2), (-2.6, 3.2), (-2.6, 1.85)], "True", label_off=(0, .22))
    _arrow(ax, [(1.5, 3.2), (2.6, 3.2), (2.6, 1.85)], "False", label_off=(0, .22))
    _arrow(ax, [(-2.6, 0.95), (-2.6, -0.3), (-2.3, -0.3)])
    _arrow(ax, [(2.6, 0.95), (2.6, -0.3), (2.3, -0.3)])
    _arrow(ax, [(0, -0.75), (0, -1.5)])
    _frame(ax, (-4.6, 4.6), (-1.8, 4.3))

def draw_flow_if_elif_else(ax):
    """Replaces the TikZ flowchart in 05_Branching (if / elif / else)."""
    ys = [6.0, 4.4, 2.8]
    for k, y in enumerate(ys):
        _diamond(ax, (0, y), ("if" if k == 0 else "elif") + " condition:")
        _box(ax, (4.4, y), "execute block #%d" % (k + 1), w=3.2)
        _arrow(ax, [(1.5, y), (2.8, y)], "True", label_off=(0, .22))
        # every block merges into a shared rail on the right; only the rail
        # itself carries an arrowhead, into "code after the else block"
        _line(ax, [(6.0, y), (7.3, y)])
    _line(ax, [(7.3, ys[0]), (7.3, -1.7)])
    _arrow(ax, [(7.3, -1.7), (2.3, -1.7)])
    for a, b in zip(ys, ys[1:]):
        _arrow(ax, [(0, a - 0.75), (0, b + 0.75)], "False", label_off=(-0.45, 0),
               ha="right")
    ax.text(0, 1.6, "$\\vdots$", ha="center", va="center", fontsize=14, color=INK)
    ax.text(0.5, 1.6, "more elif's as desired", ha="left", va="center",
            fontsize=8, color=MUTED)
    _arrow(ax, [(0, 2.05), (0, 1.9)], "False", label_off=(-0.45, 0), ha="right")
    _box(ax, (0, 0.6), "else block (optional)", w=3.4)
    _arrow(ax, [(0, 1.3), (0, 1.05)])
    _box(ax, (0, -1.7), "code after the else block", w=4.4)
    _arrow(ax, [(0, 0.15), (0, -1.25)])
    _frame(ax, (-2.6, 8.4), (-2.6, 7.0))

# ---------------------------------------------------------------- runtime ---
_SIZES = {'flow_if_else': (6.4, 4.2), 'flow_if_elif_else': (7.6, 6.4)}
_WIDTHS = {}
_FIGURES = {}


def _render(name):
    fig, ax = plt.subplots(figsize=_SIZES.get(name, (6.4, 4.4)), dpi=110)
    globals()["draw_" + name](ax)
    fig.tight_layout(pad=0.3)
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return buf.getvalue()


def show(name, width=None):
    """Display one of this chapter's figures."""
    display(Image(_FIGURES[name], width=width or _WIDTHS.get(name, 560)))


for _n in ['flow_if_else', 'flow_if_elif_else']:
    _FIGURES[_n] = _render(_n)

print("Setup complete \u2014 2 figure(s) ready.")


### Relational Operators and Boolean Expressions

An operator that produces a result that is either `True` or `False` is called a *relational operator*.  Relational operators are used to ask simple "true or false" questions about how one piece of data is *related* to another.  Thus, relational operators always have two operands.  For example, the value of the expression `2 < 4` is `True`.  This is because the `<` operator is the "less than" operator.  More generally, the expression `$x$ < $y$` has the value `True` if the value of $x$ is smaller than the value of $y$.   The following table lists several commonly used relational operators in Python.

{Operator} | Meaning | {Example} | {Result} |
|---|---|---|---|
| == | are the operands equal? | 42 == 42 | True |
| != | are the operands unequal? | 42 != 42 | False |
| < | is the first operand smaller than the second operand? | 10 < 42 | True |
| > | is the first operand larger than the second operand? | "Bill" > "Lenny" | False |
| <= | is the first operand less than or equal to the second? | 42 <= 42 | True |
| >= | is the first operand greater than or equal to the second? | "R" >= "Z" | False |

Notice how the operators work with non-numeric data as well, like strings and characaters.  In such cases the comparison is made lexicographically (dictionary ordering).  `"Bill"` is not greater than `"Lenny"` because `"Bill"` comes before `"Lenny"` in dictionary ordering.  For the same reason, the expression `"Bill" < "Lenny"` has the value `True`.

Relational operators all have the same precedence and so, are evaluated from left-to-right. But all relational operators also have a lower precedence than all arithmetic operators, which means arithmetic operators get evaluated first.  Thus the expression `5 + 5 < 10` is `False` because the addition happens first, resulting in the value 10.  Since 10 is not less than 10, the `<` operator evaluates to `False`.

#### Boolean Expressions

A *Boolean expression* is any expression whose value is either `True` or `False`.  Thus, all of the expressions in the third column of the above table are Boolean expressions.

### Logical Operators

The operators `and`, `or`, and `not` are *logical operators* (also called *Boolean operators*).  They are so-called because the operands of logical operators must be Boolean values.  Thus the operands of logical operators can either be Boolean values or other Boolean expressions.  We can use logical operators to ask questions about Boolean values or Boolean expressions.

All logical operators have a **lower** precedence than relational operators.  So that means that relational operators always get evaluated before logical operators.

#### The `and` Operator

The expression `$x$ and $y$` has a value of `True` only if both $x$ and $y$ are `True`.  In all other cases, such an expression has a value of `False`.  Remember that $x$ and $y$ could be Boolean literals, Boolean values, Boolean expressions, or even a function call that returns a Boolean value.  Here are some examples:

| Expression | Value |
|---|---|
| 1 - 1 > 0 and -2 > 0 | False |
| False and 'x' < 'y' | False |
| 9 >= 9 and "FortyTwo".isdigit() | False |
| 5 < 10 and 20 != 42 | True |
| len("Skywalker") > 0 and len("Skywalker") < 10 and "Ren" < "Rey" | True |

Note the order of operations in the first example.  The subtraction happens first, because it has higher precedence than all relational and logical operators.  Then the two greater-than operators are evaluated because relational operators have higher precedence than logical operators.  The last thing that happens is the `and` operator.  Since both `>`  operators result in `False`, the entire expression is `False`.

In the third example, we call the `isdigit` method on the string  `"FortyTwo"`.  Since the string `FortyTwo` doesn't contain digits, the function returns `False`.  Therefore, even though the relation `9 >= 9` is `True`, the entire expression has the value `False`.

In the last example, the two `and` operators are evaluated left-to-right.  The result of the first `and` is `True`, which becomes the first operand to the second `and`, then `True and "Ren" < "Rey"` evaluates to `True`, so the whole expression evaluates to `True`.

#### The `or` Operator

The expression `$x$ or $y$` has a value of `False` only if both $x$ and $y$ are `False`.  In all other cases, such an expression has a value of `True`.  Here are some examples of expressions using `or`:

| Expression | Value |
|---|---|
| 5 < 7 or 0 == 0 | True |
| 7 < 5 or 0 == 0 | True |
| 2**5 < 16 or max(7, 42) == 7 | False |
| "Skywalker".find("Anakin") > -1 or "Skywalker".islower() | False |

The last example is `False` because `"Anakin"` is not a substring of `"Skywalker"` so the `find` function returns -1. Because -1 is not greater than -1, the first operand to `or` is `False`.  `"Skywalker".islower()` is also `False` since `"Skywalker"` does not consist only of lowercase characters.  Thus, both operands are `False`, so the `or` evaluates to `False`.

#### The `not` Operator

The `not` operator is a unary operator.  It only takes one operand.  The expression `not $x$` has a value of `True` only if $x$ is `False`; it has a value of `False` if $x$ is `True`.  So `not` changes the Boolean value of its operand to the other Boolean value.  Here are some examples:

| Expression | Value |
|---|---|
| not 42 < 0 | True |
| not 6 == 6 | False |
| not max(17, 50) > 80 | True |

In the last example, the function call `max` has the highest precedence; it returns 50.  The next highest precedence is the `>` operator (relational operators have higher precedence than logical operators), which results in `False` since 50 is not greater than 80, then `not False` results in `True`.

#### Mixing Logical Operators

We don't want you to get the idea that you can only use one kind of logical operator per expression.  You can mix them up as much as you like, but take care --- **the logical operators do not have the same precedence!**  The operator `not` has higher precedence than `and` which, in turn, has higher precedence than `or`.  Take a look at these expressions:

| Expression | Value |
|---|---|
| not 5 < 7 or 0 == 0 | True |
| not (5 < 7 or 0 == 0) | False |
| len("Vader") < 7 or len("Maul") < 3 and "Vader" < "Maul" | True |
| (len("Vader") < 7 or len("Maul") < 3) and "Vader" < "Maul" | False |

You might expect the first expression to have a value of `False`, because `5 < 7 or 0 == 0 ` is clearly `True`, and the `not` would change that to `False`.   But the `not` operator has higher precedence than `or`.  In this expression, the relational operators evaluate first, giving us `not True or True`.  Now the `not` is applied to the first `True`, giving us `False or True`, which ends up as `True`.  If we really want to apply `not` to the result of the `or`, we have to add parentheses, like in the second example.  The relational operators still evaluate first, again giving us `not (True or True)`.  But now, because of the parentheses, the `or` evaluates next, which gives us `not True`, and ultimately `False`.

Note how in the third and fourth examples, if we want the `or` to evaluate before the `and` we have to use parentheses around the `or` expression.  You can see that it matters because we get different answers depending on which of `or` or `and` evaluates first.

#### Variables in Relational and Logical Expressions

We also don't want you to get the idea that you can't use variables with these operators.  In any of the examples above where a literal appears in an expression, we could also replace the literal with a variable.  For example, `a < b and c < d`.  We just can't evaluate this without knowing the values of the variables.  Here's a complete example where we associate the variable names with values and use them in a Boolean expression:

In [ ]:
a = 1
b = 5
c = 2
d = 4
a < b and c < d

### Branching and Conditional Statements

Now that we know how to ask questions about data using Boolean expressions, we can use the values of Boolean expressions to get our programs to perform different actions depending on the value of a Boolean expression.  This is called *branching* and it allows us to perform one block of code if a Boolean expression is `True`, and a different one if it is `False`.

In Python, we perform branching using a *conditional statement* or *if-statement*.  The syntax is the word `if`, followed by a Boolean expression, followed by a colon, like this:

> `if {*condition*}:`

The if-statement is then followed by a *block*.  Remember blocks from Chapter ?  A block is a series of indented lines of code.  The block of code following the if-statement is only executed if the condition in the if-statement evaluates to `True`.  Let's look at an example:

In [ ]:
# ▶ Interactive: this cell waits for you to type something.
# Run it yourself - "Run All" skips it so the rest of the chapter still works.
guess = int(input('Guess a number between 1 and 100'))
if guess >= 1 and guess <= 100:
	print('That was a valid guess!')

The first line of this example asks the user to input a number between 1 and 100.  The name `guess` is assigned to the value entered.  Then we have an if-statement.  The condition of the if-statement is the Boolean expression `guess >= 1 and guess <= 100`.  The value of this expression will, of course, depend on the value of `guess`.  If `guess` is, in fact, between 1 and 100, the Boolean expression is `True`, and the one-line block of code consisting of the `print` call is executed.  Otherwise, it is not.  Here is what we see if we run the program, and enter the number 50 (green text is text entered by a user):

```

Guess a number between 1 and 100: 50
That was a valid guess!

```

Since the Boolean expression in the if-statement is `True`, the indented block consisting of the call to `print` is executed.  If we enter a value that is not between 1 and 100, the `print` call will not execute and we will not see the output `That was a valid guess!`.  But what if we want to print something different if the guess is not between 1 and 100?  It might be natural to try this:

In [ ]:
# ▶ Interactive: this cell waits for you to type something.
# Run it yourself - "Run All" skips it so the rest of the chapter still works.
guess = int(input("Guess a number between 1 and 100: "))
if guess >= 1 and guess <= 100:
	print("That was a valid guess!")
	
print("That was not a valid guess.")

But this won't work because the second call to `print` will execute regardless of whether the Boolean expression in the if-statement is `True`.  What we need is a way of specifying a second block that gets executed only if the Boolean expression in the if-statement is `False`.  We can do this using an *else-statement*.  An else-statement is the word `else` followed by a colon:

In [ ]:
# ▶ Interactive: this cell waits for you to type something.
# Run it yourself - "Run All" skips it so the rest of the chapter still works.
guess = int(input('Guess a number between 1 and 100: '))
if guess >= 1 and guess <= 100:
	# This block executes if the condition is True
	print("That was a valid guess!")
else:	
	# This block executes if the condition is False
	print("That was not a valid guess.")

Now, if we enter a number that is between 1 and 100, it will execute the first block of code.  Otherwise, it will execute the else statement's block of code.  In general, the flow of execution for conditional statements looks like this:

In [ ]:
show("flow_if_else")

```python
if condition:
	# if block (indented)
else:
	# else block (indented)
	
## code after else block
```

Now suppose we wanted to give the user a little more information about why a guess was invalid.  If the user guessed a number that was too large, we want to print out `Too high!`.  If they guess too low, we want to print out `Too low!`.  Otherwise, we want to print out `That was a valid guess`.  Here's one way we could do that:

In [ ]:
# ▶ Interactive: this cell waits for you to type something.
# Run it yourself - "Run All" skips it so the rest of the chapter still works.
guess = int(input('Guess a number between 1 and 100: '))
if guess < 1:
	print('Too low!')
	
if guess > 100:
	print('Too high!')
	
if guess >= 1 and guess <= 100:
	print('That was a valid guess!')

But Python, and most other programming languages give us a cleaner way to do this that guarantees that only one of a series of blocks can be executed.  In Python, there is an elif-statement ("elif" is short for "else if").   An elif-statement consists of the word "elif", followed by a Boolean expression, followed by a colon, followed by a block of statements to execute if the Boolean expression is `True`.
An elif-statement can appear after the block associated with an if-statement or another elif-statement, but is only executed if the preceding if- or elif-statement was found to be `False`.  So here's a different way we could write our program which does the same thing, but is a bit easier to read:

{

In [ ]:
# ▶ Interactive: this cell waits for you to type something.
# Run it yourself - "Run All" skips it so the rest of the chapter still works.
guess = int(input('Guess a number between 1 and 100: '))
if guess < 1:
	# If guess was less than one, execute this block.
	print('Too low!')
elif guess > 100:
	# Otherwise, if guess is larger than 100, do this block.
	print('Too high!')
else:	
	# Otherwise, execute this block.
	print('That was a valid guess!')

Note that only one of the three blocks is executed.  As soon as an if- or elif- statement is `True`, its block is executed and no more if- or elif- statement conditions are tested, and no more of the blocks can execute.  The final else block only executes if none of the preceding conditions were `True`.  Once one of the blocks executes, the execution continues at the first line of code following the else block.  Multiple elif-statements and accompanying blocks are allowed as long as the first conditional statement is an if-statement.  In all cases the else statement is optional.  The flow of execution in an if-elif-else chain is described by the following flowchart and code template:

In [ ]:
show("flow_if_elif_else")

```python
if condition:
	# block 1 (indented)
elif condition:
	# block 2 (indented)
elif condition:
	# block 3 (indented)

## ... more elif's as desired

else:  # (optional)
	# else block

## code after the else block
```

Notice that only one of the blocks in the if-elif-elif-...-else chain can execute no matter how many elif-statements there are.  Finally, remember that the blocks can consist of multiple lines of code, as long as they are all indented.

In [ ]:
smaller = 9    # try swapping these two values
larger = 4

if smaller > larger:
    # swap the values referred to by the variables
    temp = smaller
    smaller = larger
    larger = temp

print(smaller, larger)

Because all three lines after the if-statement in the above code are indented, they are all part of the block, and all three only get executed if the if-statement's condition is `True`. Block indentation must be such that every line of every block is indented by the same amount, otherwise Python will not understand your program.

#### Nested branches

It is often necessary or useful to include an if statement inside another, this is called a nested conditional or nested branch. For example, imagine that we are deciding whether to go to the beach or not. There are many beautiful beaches on PEI, but PEI is also quite windy. So we want to go to the beach if the temperature is high enough (say above $20\degree$ C) and the day is not windy. We can easily write a program to take this decision, and we want our program to differentiate whether we are not going to the beach because of the temperature or the wind.

In [ ]:
temperature = 25   # try changing these two values
windy = False

if temperature > 20:
    if not windy:
        print("Perfect weather, lets go to the beach!")
    else:
        print("The temperature is nice, but it is too windy.")
else:
    print("It is too cold, I'll stay home.")